# Thai ASR: three models, pairwise agreement and best text

```
Audio segment files from VAD
   ├── distill  ─► hypothesis text ─┐
   ├── typhoon  ─► hypothesis text ─┼─► normalization (+ PyThaiNLP word tokenization for WER_norm)
   └── pathumma ─► hypothesis text ─┘        │
                                             ▼
        compare every pair: CER_raw, CER_norm, WER_norm
          distill <-> pathumma, distill <-> typhoon, pathumma <-> typhoon
                                             │
                                             ▼
        Mean_Score(A) = (CER_norm(A,B) + CER_norm(A,C)) / 2   ─►  lowest score  ─►  best text
```

There is no reference transcript: the three models are scored against each other. A model that agrees with the
other two has a low mean score, so its transcript is taken as the best text.

**Output**: `results.jsonl`, one line per audio segment, in the same format as `results-parallel.jsonl`.
Steps 1 to 6 produce it; step 7 is an optional summary.

**How to run**
1. Put the VAD segment `.wav` files on Google Drive and set `AUDIO_DIR` in the config cell.
2. `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`, and approve the Google Drive prompt.
3. If Colab disconnects, run all again. Finished segments are skipped.

In [ ]:
!pip -q install pythainlp jiwer soundfile librosa

## 1. Config

In [ ]:
import os, glob, json, gc
import numpy as np
import torch

MODELS = {
    "distill":  "biodatlab/distill-whisper-th-large-v3",
    "pathumma": "nectec/Pathumma-whisper-th-large-v3",
    "typhoon":  "typhoon-ai/typhoon-whisper-large-v3",
}
BATCH_SIZE = 16
SR         = 16000
MAX_SEC    = 30.0     # longer segments are transcribed one at a time in long-form mode

from google.colab import drive
drive.mount("/content/drive")
AUDIO_DIR = "/content/drive/MyDrive/vad_segments"     # folder of VAD segment .wav files (subfolders are searched too)
OUT_DIR   = "/content/drive/MyDrive/thai_asr_score"
os.makedirs(OUT_DIR, exist_ok=True)

HYP_JSONL    = {m: f"{OUT_DIR}/hyp_{m}.jsonl" for m in MODELS}   # one model's transcripts, appended batch by batch
RESULTS_JSONL = f"{OUT_DIR}/results.jsonl"                       # final output, one line per segment

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "NO GPU - switch the runtime to T4")

## 2. Audio segment files from VAD

`audio_path` in the output is the file name only, as in `results-parallel.jsonl`, so file names must be unique.

In [ ]:
paths = sorted(glob.glob(f"{AUDIO_DIR}/**/*.wav", recursive=True))
SEGMENTS = {os.path.basename(p): p for p in paths}
assert len(SEGMENTS) == len(paths), "two segments share a file name; audio_path would be ambiguous"
print(f"{len(SEGMENTS)} segments in {AUDIO_DIR}")

## 3. Transcribe with each model

Same settings for all three models: fp16, greedy decoding, `language="th"`, `task="transcribe"`, one model on the
GPU at a time. Segments of 30 s or less are transcribed in batches. Longer ones are transcribed one at a time in
Whisper's long-form mode, so nothing after 30 s is cut off.

Each model writes `hyp_<model>.jsonl` (`{"audio_path", "hyp"}` or `{"audio_path", "error"}`) after every batch.
A segment that cannot be read or transcribed gets its error message instead of a transcript, like the
`errors` field in the output.

In [ ]:
import soundfile as sf, librosa
from tqdm.auto import tqdm
from transformers import WhisperForConditionalGeneration, WhisperProcessor

def load_audio(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim > 1: y = y.mean(axis=1)
    if sr != SR: y = librosa.resample(y, orig_sr=sr, target_sr=SR)
    return y

def load_model(name):
    proc = WhisperProcessor.from_pretrained(MODELS[name])
    model = WhisperForConditionalGeneration.from_pretrained(MODELS[name], torch_dtype=DTYPE).to(DEVICE).eval()
    model.generation_config.forced_decoder_ids = None      # language and task are passed explicitly below
    return model, proc

@torch.inference_mode()
def transcribe(model, proc, waves):
    long = max(len(w) for w in waves) > MAX_SEC * SR
    if long:   # long-form: keep the whole segment, Whisper decodes it 30 s at a time
        feats = proc(waves, sampling_rate=SR, return_tensors="pt", truncation=False, padding="longest", return_attention_mask=True)
    else:
        feats = proc(waves, sampling_rate=SR, return_tensors="pt", return_attention_mask=True)
    ids = model.generate(
        input_features=feats.input_features.to(DEVICE, DTYPE),
        attention_mask=feats.attention_mask.to(DEVICE),
        language="th", task="transcribe", return_timestamps=long, num_beams=1, do_sample=False,
    )
    return [t.strip() for t in proc.tokenizer.batch_decode(ids, skip_special_tokens=True)]

def read_jsonl(path):
    if not os.path.exists(path): return []
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def append_jsonl(path, rows):
    with open(path, "a", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

def run_model(name):
    done = {r["audio_path"] for r in read_jsonl(HYP_JSONL[name])}
    todo = [a for a in SEGMENTS if a not in done]
    if not todo:
        print(f"{name}: already done"); return
    model, proc = load_model(name)
    lengths = {a: sf.info(SEGMENTS[a]).duration if os.path.exists(SEGMENTS[a]) else 0 for a in todo}
    short = [a for a in todo if lengths[a] <= MAX_SEC]
    batches = [short[i:i + BATCH_SIZE] for i in range(0, len(short), BATCH_SIZE)] + [[a] for a in todo if lengths[a] > MAX_SEC]
    for batch in tqdm(batches, desc=name):
        rows, waves, ok = [], [], []
        for a in batch:
            try: waves.append(load_audio(SEGMENTS[a])); ok.append(a)
            except Exception as e: rows.append({"audio_path": a, "error": str(e)})
        if ok:
            try:
                rows += [{"audio_path": a, "hyp": h} for a, h in zip(ok, transcribe(model, proc, waves))]
            except Exception:                       # find which segment failed: retry one by one
                for a, w in zip(ok, waves):
                    try: rows.append({"audio_path": a, "hyp": transcribe(model, proc, [w])[0]})
                    except Exception as e: rows.append({"audio_path": a, "error": str(e)})
        append_jsonl(HYP_JSONL[name], rows)
    del model; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

for name in MODELS:
    run_model(name)
print("transcription finished")

## 4. Normalization functions (for CER_norm and WER_norm)

Needs no GPU, so this and the next steps can be rerun at any time.

**Normalization.** The three models write the same speech in different styles, such as spaces, ๆ, punctuation
and letter case. These are formatting differences, not recognition errors. Normalization removes them, so the
score measures only real differences in content. The same steps are applied to the reference and to every model's output.

- **Lower case**: English words can be written as TikTok or tiktok, so everything is converted to lower case.
- **Thai character normalization**: `pythainlp.util.normalize` fixes characters typed in a different order or
  duplicated, which look the same but are different strings (`เเ` becomes `แ`, `นํ้า` becomes `น้ำ`).
- **ๆ (maiyamok)**: ๆ means the word before it is spoken twice, so just deleting it would break the match.
  `pythainlp.util.maiyamok` first expands it into the repeated word (`คนๆนก` becomes `คน คน นก`).
- **Number words to digits**: spelled-out numbers become digits, so a model that writes numbers as words
  (typhoon never writes digits) is not penalised. Applied to every model and the reference. It uses PyThaiNLP's
  `thaiword_to_num` on whole `newmm` tokens only:
  - ordinary numbers: `สองพันสิบ` becomes `2010`, `เก้าร้อยแปดสิบจุดเก้าห้า` becomes `980.95`
  - digits read one by one (two or more in a row): `ศูนย์แปดหนึ่งสอง` becomes `0812`
  - words that only contain a number word, such as `ห้าม` or `เก้าอี้`, are left alone
  - a lone `ศูนย์`, `ร้อย`, `พัน`, `หมื่น`, `แสน` or `ล้าน` is left as written (`ศูนย์` usually means centre)
  - known limits: `หนึ่ง` used as "a/one" becomes `1` (on both sides, so it mostly cancels), and mixed forms such
    as `100 ล้าน` against `ร้อยล้าน` still count as different
- **Punctuation**: models write symbols differently, so all of `string.punctuation` plus the Thai symbol ฯ
  are removed (ๆ is already expanded in the step above). The one exception is a decimal point between two digits, so `980.95` does not turn into `98095`.
- **Space**: each model puts spaces in different places, so all whitespace is removed.
- **Word tokenizer** (WER only): Thai has no spaces between words, so the normalised text is split into words
  with PyThaiNLP `newmm`.

In [ ]:
import re, string
from pythainlp.util import maiyamok, thaiword_to_num, normalize as th_normalize
from pythainlp.tokenize import word_tokenize

# --- Normalization: removes formatting differences (case, Thai char order, ๆ, number words, punctuation, spaces) ---
# all punctuation and ฯ, except a decimal point between digits (980.95 stays 980.95).
# ๆ is not listed: expand_maiyamok has already turned it into the repeated word.
SYMBOLS = re.compile(r"(?!(?<=\d)\.(?=\d))[ฯ%s]" % re.escape(string.punctuation))

def remove_space(s):
    return "".join(s.split())

def expand_maiyamok(s):
    "ๆ means the word before it is spoken twice: คนๆนก -> คนคนนก"
    return "".join(maiyamok(s))

def remove_punctuation(s):
    return SYMBOLS.sub("", s)

def to_lower(s):
    return s.lower()

DIGIT_WORDS = {"ศูนย์": "0", "หนึ่ง": "1", "สอง": "2", "สาม": "3", "สี่": "4", "ห้า": "5", "หก": "6", "เจ็ด": "7", "แปด": "8", "เก้า": "9"}
BARE_UNITS  = {"ศูนย์", "ร้อย", "พัน", "หมื่น", "แสน", "ล้าน"}   # left as written when they stand alone

def is_number_word(tok):
    "True only when the whole token is a Thai number word, so ห้าม or เก้าอี้ are never touched"
    if tok in DIGIT_WORDS: return True
    if "ลบ" in tok: return False                                 # ลบ is more often "delete" than "minus"
    try: thaiword_to_num(tok); return True
    except Exception: return False

def convert_number_run(run):
    "run = consecutive number-word tokens, possibly ending in จุด plus decimal digits"
    if len(run) == 1 and run[0] in BARE_UNITS:                   # ศูนย์ = centre, หลายร้อย, แสนสาหัส, 3.5 ล้าน
        return run[0]
    if len(run) >= 2 and all(t in DIGIT_WORDS for t in run):     # read digit by digit: phone numbers, codes
        return "".join(DIGIT_WORDS[t] for t in run)
    try:
        if "จุด" in run:
            k = run.index("จุด")
            frac = "".join(DIGIT_WORDS[t] for t in run[k + 1:])  # decimals are read digit by digit
            return f"{thaiword_to_num(''.join(run[:k]))}.{frac}"
        return str(thaiword_to_num("".join(run)))
    except Exception:
        return "".join(run)                                      # not a well-formed number: leave as written

def numbers_to_digits(s):
    toks, out, i = word_tokenize(s, engine="newmm"), [], 0
    while i < len(toks):
        j, decimal = i, False
        while j < len(toks):
            t = toks[j]
            if decimal: ok = t in DIGIT_WORDS                    # after จุด only single digits belong to the number
            elif t == "จุด": ok = decimal = j > i and j + 1 < len(toks) and toks[j + 1] in DIGIT_WORDS
            else: ok = is_number_word(t)
            if not ok: break
            j += 1
        if j > i: out.append(convert_number_run(toks[i:j])); i = j
        else: out.append(toks[i]); i += 1
    return "".join(out)

def normalize_text(s):
    s = to_lower(str(s))
    s = th_normalize(s)           # fix Thai character order and duplicated vowels/tone marks (เเ -> แ, นํ้า -> น้ำ)
    s = expand_maiyamok(s)        # before removing symbols, so ๆ becomes the repeated word
    s = numbers_to_digits(s)      # สองพันสิบ -> 2010, so a model that spells numbers out (typhoon) is not penalised
    s = remove_punctuation(s)
    return remove_space(s)

def tokenize(s):
    "Thai has no spaces between words, so split with PyThaiNLP newmm (used for WER only)"
    return [t for t in word_tokenize(s, engine="newmm", keep_whitespace=False) if t.strip()]

# --- Scoring with jiwer ---
def jiwer_counts(ref, hyp, unit):
    "substitutions, deletions, insertions, reference length. unit = 'char' (CER) or 'word' (WER, token lists)"
    if unit == "word":
        ref, hyp = " ".join(ref), " ".join(hyp)         # jiwer splits words on spaces
    if not ref.strip():                                  # jiwer rejects an empty reference
        return 0, 0, len(hyp.split() if unit == "word" else hyp), 0
    o = (jiwer.process_words if unit == "word" else jiwer.process_characters)(ref, hyp)
    return o.substitutions, o.deletions, o.insertions, o.hits + o.substitutions + o.deletions

def has_repetition(tokens, n=4, times=3):
    grams = {}
    for k in range(len(tokens) - n + 1):
        g = tuple(tokens[k:k + n]); grams[g] = grams.get(g, 0) + 1
    return bool(grams) and max(grams.values()) >= times

def score_row(ref, hyp):
    hyp = "" if pd.isna(hyp) else str(hyp)
    out = {}
    rn, hn = normalize_text(ref), normalize_text(hyp)
    rr, hr = str(ref).strip(), hyp.strip()
    rn_tok, hn_tok = tokenize(rn), tokenize(hn)
    for tag, r, h, unit in [("cer", rn, hn, "char"), ("wer", rn_tok, hn_tok, "word"),
                            ("cer_raw", rr, hr, "char"), ("wer_raw", tokenize(rr), tokenize(hr), "word")]:
        s, d, i, n = jiwer_counts(r, h, unit)
        out.update({f"{tag}_sub": s, f"{tag}_del": d, f"{tag}_ins": i, f"{tag}_n": n,
                    f"{tag}_err": s + d + i, tag: (s + d + i) / max(n, 1)})
    out["ref_norm"], out["hyp_norm"] = rn, hn
    out["len_ratio"]     = len(hn) / max(len(rn), 1)
    out["hallucination"] = out["len_ratio"] > 1.5 or (has_repetition(hn_tok) and not has_repetition(rn_tok))
    out["catastrophic"]  = out["cer"] > 0.5
    out["empty_output"]  = len(hn) == 0
    return out

## 5. Normalize every hypothesis

Loads each model's transcripts and runs the normalization once per transcript:
- `norm[model][audio_path]`: normalized text, used for CER_norm
- `words[model][audio_path]`: the normalized text split into PyThaiNLP `newmm` words, used for WER_norm

Only segments not yet in `results.jsonl` are processed, so a rerun skips finished work.

In [ ]:
# load every model's transcripts
hyps = {m: {} for m in MODELS}     # hyps[model][audio_path] = transcript as written
errs = {m: {} for m in MODELS}     # errs[model][audio_path] = error message
for m in MODELS:
    for r in read_jsonl(HYP_JSONL[m]):
        if "hyp" in r: hyps[m][r["audio_path"]] = r["hyp"]; errs[m].pop(r["audio_path"], None)
        else:          errs[m][r["audio_path"]] = r["error"]; hyps[m].pop(r["audio_path"], None)

# segments still to score: every model has finished them and they are not in results.jsonl yet
done = {r["audio_path"] for r in read_jsonl(RESULTS_JSONL)}
todo = [a for a in SEGMENTS if a not in done and all(a in hyps[m] or a in errs[m] for m in MODELS)]
print(f"{len(done)} segments already in results.jsonl, {len(todo)} to normalize and score")

# normalize every transcript once (for CER_norm) and split it into words (for WER_norm)
norm  = {m: {} for m in MODELS}    # norm[model][audio_path]  = normalized text
words = {m: {} for m in MODELS}    # words[model][audio_path] = newmm word list of the normalized text
for m in MODELS:
    for a in tqdm([a for a in todo if a in hyps[m]], desc=f"normalize {m}"):
        norm[m][a] = normalize_text(hyps[m][a])
        words[m][a] = tokenize(norm[m][a])

# check one segment
if todo:
    a = next((a for a in todo if all(a in hyps[m] for m in MODELS)), todo[0])
    print("\n", a)
    for m in MODELS:
        if a in hyps[m]: print(f"  {m:9s} raw : {hyps[m][a]}\n  {'':9s} norm: {norm[m][a]}\n  {'':9s} words: {words[m][a]}")

## 6. Score: compare the hypotheses and pick the best text

Uses the normalized text and words from step 5; nothing is normalized again here.

For every segment, each pair of models is scored. The model that comes first alphabetically is the reference:

| Score | Text used |
|---|---|
| `cer_raw` | the transcripts as written, outer whitespace trimmed |
| `cer_norm` | the normalized transcripts |
| `wer_norm` | the normalized transcripts split into PyThaiNLP `newmm` words |

Then each model gets `Mean_Score(A) = (CER_norm(A,B) + CER_norm(A,C)) / 2` (`mean_agreement_per_model`).
The lowest mean score is the best model; a tie goes to the model first in alphabetical order.
`best_text` (and `text`) is the best model's transcript with all spaces removed. `unanimous` is true when every
`cer_norm` is 0. All scores are rounded to 4 decimals.

If a model fails on a segment, its hypothesis is `null` and its error message goes in `errors`. Fewer than two
transcripts cannot be compared, so then `pairs` is empty, `best_model` and `best_text` are `null` and `text` is `""`.

In [ ]:
import itertools
import jiwer

def cer(ref, hyp):
    "character error rate with ref as the reference; jiwer rejects an empty reference, so that case is handled here"
    if not ref: return 0.0 if not hyp else 1.0
    return jiwer.cer(ref, hyp)

def wer(ref_words, hyp_words):
    "ref_words / hyp_words are newmm word lists; jiwer splits words on spaces"
    if not ref_words: return 0.0 if not hyp_words else 1.0
    return jiwer.wer(" ".join(ref_words), " ".join(hyp_words))

def compare(raw, nrm, wrd):
    """raw / nrm / wrd = {model: raw text / normalized text / word list} for the models that succeeded.
    Every pair is scored with the first model (alphabetical) as the reference. Mean_Score(A) = mean of CER_norm
    over the pairs that contain A; with three models that is (CER_norm(A,B) + CER_norm(A,C)) / 2.
    Lowest mean = best model (ties: alphabetical)."""
    names = sorted(raw)
    if len(names) < 2:            # nothing to compare against
        return {"pairs": [], "mean_agreement_per_model": {}, "unanimous": False, "best_model": None, "best_text": None}
    pairs = []
    for a, b in itertools.combinations(names, 2):
        pairs.append({"pair": [a, b],
                      "cer_raw":  round(cer(raw[a].strip(), raw[b].strip()), 4),
                      "cer_norm": round(cer(nrm[a], nrm[b]), 4),
                      "wer_norm": round(wer(wrd[a], wrd[b]), 4)})
    mean = {}
    for m in names:
        own = [p["cer_norm"] for p in pairs if m in p["pair"]]
        mean[m] = round(sum(own) / len(own), 4)
    best = min(names, key=lambda m: mean[m])
    return {"pairs": pairs, "mean_agreement_per_model": mean,
            "unanimous": all(p["cer_norm"] == 0 for p in pairs),
            "best_model": best, "best_text": "".join(raw[best].split())}

def make_record(a):
    ok = [m for m in MODELS if a in hyps[m]]
    comp = compare({m: hyps[m][a] for m in ok}, {m: norm[m][a] for m in ok}, {m: words[m][a] for m in ok})
    return {"audio_path": a,
            "text": comp["best_text"] or "",
            "hypotheses": {m: hyps[m].get(a) for m in sorted(MODELS)},
            "errors": {m: errs[m][a] for m in sorted(MODELS) if a in errs[m]},
            "comparison": comp}

buf = []
for a in tqdm(todo, desc="score"):
    buf.append(make_record(a))
    if len(buf) >= 1000: append_jsonl(RESULTS_JSONL, buf); buf = []
append_jsonl(RESULTS_JSONL, buf)
print("results:", RESULTS_JSONL)

## 7. Compare scores (optional)

This step only reads `results.jsonl` and prints summary tables. It writes nothing, so you can skip it and
`results.jsonl` stays the same. It is a quick sanity check: one model winning almost every segment, or a very low
unanimous share, would suggest something went wrong.

In [ ]:
import pandas as pd

res = read_jsonl(RESULTS_JSONL)
comp = [r["comparison"] for r in res if r["comparison"]["best_model"] is not None]
print(f"{len(res)} segments, {len(comp)} compared, {len(res) - len(comp)} without a best text (errors)")

print("\nhow often each model gives the best text")
display(pd.Series([c["best_model"] for c in comp]).value_counts().rename("segments").to_frame()
          .assign(percent=lambda d: (100 * d.segments / d.segments.sum()).round(1)))

print("\nmean Mean_Score per model (lower = agrees more with the other two)")
display(pd.DataFrame([c["mean_agreement_per_model"] for c in comp]).mean().round(4).rename("mean score").to_frame())

print("\nmean pairwise scores")
pairs = pd.DataFrame([{"pair": " <-> ".join(p["pair"]), **{k: p[k] for k in ("cer_raw", "cer_norm", "wer_norm")}}
                      for c in comp for p in c["pairs"]])
display(pairs.groupby("pair").mean().round(4))
print(f"\nunanimous (all three agree after normalization): {100 * np.mean([c['unanimous'] for c in comp]):.1f}% of segments")

print("\nexample lines")
for r in res[:3]: print(json.dumps(r, ensure_ascii=False)[:400])